# 02 - Connect VPN

Generate VPN client profile and verify private DNS resolution over the active VPN connection.


## Overview

End state: laptop connected with Azure VPN Client and service hostnames resolving to private endpoint IPs.


## Ensure DNS forwarder VM is running

The redeploy-safe Bicep skips the DNS forwarder VM when it already exists (see `01_deploy_infra.ipynb`), so Bicep will **not** wake it up on its own. If the VM was deallocated (for example to save cost overnight), the forwarder at `DNS_FORWARDER_PRIVATE_IP` is unreachable and every DNS query over the VPN times out — which typically presents as "internet dies when VPN connects" because the Azure VPN Client on Windows installs a catch-all NRPT rule pointing at that DNS server.

The next cell starts the VM if it is stopped/deallocated and is a no-op if it is already running.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from dotenv import load_dotenv

demo_root = Path("..").resolve()
if str(demo_root) not in sys.path:
    sys.path.insert(0, str(demo_root))

from app.notebook_helpers import resolve_az_cli

az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError("Azure CLI not found.")

load_dotenv("../env/.env")
resource_group = os.getenv("AZURE_RESOURCE_GROUP")
vm_name = "vm-search01-dns"
if not resource_group:
    raise RuntimeError("Missing AZURE_RESOURCE_GROUP in env/.env")

power_state = subprocess.run(
    [az_cmd, "vm", "get-instance-view", "--resource-group", resource_group, "--name", vm_name,
     "--query", "instanceView.statuses[?starts_with(code, 'PowerState/')].code | [0]", "-o", "tsv"],
    stdout=subprocess.PIPE, text=True, check=True,
).stdout.strip()

print(f"DNS forwarder VM power state: {power_state or '<unknown>'}")

if power_state != "PowerState/running":
    print("Starting VM (this typically takes ~60s)...")
    subprocess.run(
        [az_cmd, "vm", "start", "--resource-group", resource_group, "--name", vm_name],
        check=True,
    )
    print("✓ VM started")
else:
    print("✓ VM already running")


In [ ]:
import os
import re
import subprocess
import sys
import zipfile
from pathlib import Path

import requests
from dotenv import load_dotenv

demo_root = Path("..").resolve()
if str(demo_root) not in sys.path:
    sys.path.insert(0, str(demo_root))

from app.notebook_helpers import resolve_az_cli

az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError("Azure CLI not found.")

load_dotenv("../env/.env")
gateway = os.getenv("VNET_GATEWAY_NAME")
resource_group = os.getenv("AZURE_RESOURCE_GROUP")
dns_ip = os.getenv("DNS_FORWARDER_PRIVATE_IP")
if not gateway or not resource_group or not dns_ip:
    raise RuntimeError("Missing VNET_GATEWAY_NAME, AZURE_RESOURCE_GROUP, or DNS_FORWARDER_PRIVATE_IP in env/.env")

out_dir = Path("../outputs/vpn-client")
out_dir.mkdir(parents=True, exist_ok=True)
zip_path = out_dir / "vpnclient.zip"
# The Azure CLI no longer accepts --authentication-method AADAuthentication; the
# AAD profile flavor is governed by vpnClientConfiguration on the gateway. The
# generate API now returns the URL as a top-level string (not an object with a
# profileUrl property), so we use -o tsv directly without --query.
profile_url = subprocess.run([az_cmd, "network", "vnet-gateway", "vpn-client", "generate", "--resource-group", resource_group, "--name", gateway, "--processor-architecture", "Amd64", "-o", "tsv"], stdout=subprocess.PIPE, text=True, check=True).stdout.strip()
if not profile_url:
    raise RuntimeError("No VPN profile URL returned.")
response = requests.get(profile_url, timeout=120)
response.raise_for_status()
zip_path.write_bytes(response.content)
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(out_dir)
xml_path = out_dir / "AzureVPN" / "azurevpnconfig.xml"
if not xml_path.exists():
    raise FileNotFoundError(xml_path)

# Inject the in-VNet DNS forwarder into the profile XML, SCOPED via <dnssuffixes>
# to only the Azure Storage / AI Search FQDN spaces. Without suffix scoping the
# Azure VPN Client sends ALL DNS queries to the forwarder while connected, which
# breaks general internet DNS if the forwarder is slow or unreachable. With
# scoping, Windows installs NRPT rules only for the listed suffixes and every
# other query keeps using the local resolver (classic split-DNS).
#
# AAD-authenticated P2S VPN has no gateway-side DNS push (confirmed against the
# REST API) and the Azure VPN Client UI on Windows does not expose a DNS field,
# so this XML injection is the only sustainable mechanism. See:
# https://learn.microsoft.com/azure/vpn-gateway/azure-vpn-client-optional-configurations#dns
#
# The Azure VPN Client uses a strict DataContractSerializer that requires
# <dnsservers> before <dnssuffixes>. We do a plain text replace of the
# <clientconfig i:nil="true" /> placeholder rather than parsing+rewriting the
# XML (which reorders attributes and breaks the parser).
scoped_suffixes = [
    ".blob.core.windows.net",
    ".search.windows.net",
]
suffix_xml = "".join(f"<dnssuffix>{s}</dnssuffix>" for s in scoped_suffixes)
dns_block = (
    "<clientconfig>"
    f"<dnsservers><dnsserver>{dns_ip}</dnsserver></dnsservers>"
    f"<dnssuffixes>{suffix_xml}</dnssuffixes>"
    "</clientconfig>"
)
xml_text = xml_path.read_text(encoding="utf-8")
placeholder = re.compile(r"<clientconfig\s+i:nil=\"true\"\s*/>")
existing_block = re.compile(r"<clientconfig>.*?</clientconfig>", re.DOTALL)
if placeholder.search(xml_text):
    xml_text = placeholder.sub(dns_block, xml_text, count=1)
elif existing_block.search(xml_text):
    # Already has a clientconfig block (re-run); replace the whole block so we
    # get the current dnsservers + dnssuffixes together in the correct order.
    xml_text = existing_block.sub(dns_block, xml_text, count=1)
else:
    raise RuntimeError("Could not locate <clientconfig> in profile XML; schema may have changed.")
xml_path.write_text(xml_text, encoding="utf-8")
print(f"VPN profile XML (DNS server {dns_ip} scoped to {', '.join(scoped_suffixes)}): {xml_path.resolve()}")


## Install Azure VPN Client

- Windows: install from Microsoft Store (Windows 10 build 17763+).
- macOS: install Azure VPN Client from App Store.


## Import the profile

1. Open Azure VPN Client.
2. Select **+** then **Import**.
3. Choose `azurevpnconfig.xml` generated above.
4. Save the profile.


## How DNS is wired

For AAD-authenticated point-to-site VPN, **the gateway cannot push DNS** and the **Azure VPN Client UI does not expose a DNS field**. The only supported mechanism is editing `azurevpnconfig.xml` to add a `<clientconfig><dnsservers>` block, which the previous cell did automatically using `DNS_FORWARDER_PRIVATE_IP`.

When connected, the client installs an NRPT (Name Resolution Policy Table) rule routing queries to the forwarder. Because of NRPT, the VPN DNS server does **not** appear in `ipconfig /all` — use `Get-DnsClientNrptPolicy` to inspect it.


## Connect

Click **Connect**, sign in with AAD, and wait for profile state **Connected**.


In [ ]:
import ipaddress
import os
import socket
from dotenv import load_dotenv

load_dotenv("../env/.env")
storage_name = os.getenv("STORAGE_ACCOUNT_NAME")
search_name = os.getenv("SEARCH_SERVICE_NAME")
pe_prefix = os.getenv("SNET_PE_PREFIX")
if not storage_name or not search_name or not pe_prefix:
    raise RuntimeError("Missing STORAGE_ACCOUNT_NAME, SEARCH_SERVICE_NAME, or SNET_PE_PREFIX in env/.env")
network = ipaddress.ip_network(pe_prefix)
targets = [f"{storage_name}.blob.core.windows.net", f"{search_name}.search.windows.net"]
for host in targets:
    ip = socket.gethostbyname(host)
    print(f"{host} -> {ip}")
    if ipaddress.ip_address(ip) not in network:
        raise RuntimeError(f"{host} resolved to {ip}, not in {pe_prefix}. Revisit Edit custom DNS step and ensure VPN is connected.")
print("✓ DNS verification passed")


## Troubleshooting

- AAD sign-in popup blocked: allow popups and retry.
- Public IP resolution while connected: re-import the profile (DNS injection may have been missed) and reconnect. Confirm with `Get-DnsClientNrptPolicy` that an entry exists pointing to `DNS_FORWARDER_PRIVATE_IP`.
- DNS timeouts to `10.50.2.4`: forwarder VM cloud-init failed; the running VM has been patched but a fresh deploy needs the updated Bicep `customData` (disables systemd-resolved before installing dnsmasq).
- Consent error: run consent check from `00_setup.ipynb`.
- Drops under concurrency: gateway SKU may need scaling.


---

Continue with `03_configure.ipynb`.
